In [26]:
# Make data for learning analysis

import os
import sys
import pandas as pd
import yaml 
from matplotlib import pyplot as plt
from matplotlib import ticker as mticker
from matplotlib import colors as mcolors
from matplotlib import patches as mpatches
import statsmodels.api as sm
import numpy as np
from itertools import product
import subprocess

np.seterr(divide='ignore', invalid='ignore') # ignore divide by zero warnings

plt.rcParams['font.family'] = 'Times New Roman'
plt.rcParams['font.size'] = 11
#plt.rcParams['text.usetex'] = True

from sklearn.decomposition import PCA

with open("../../config.yaml.local", "r") as f:
    LOCAL_CONFIG = yaml.safe_load(f)
with open("../../config.yaml", "r") as f:
    CONFIG = yaml.safe_load(f)
sys.path.append("../python")

import globals
import data_tools as dt
import writing_tools as wt
import utils
#import emb

LOCAL_PATH = LOCAL_CONFIG["LOCAL_PATH"]
RAW_DATA_PATH = LOCAL_CONFIG["RAW_DATA_PATH"]
DATA_PATH = LOCAL_CONFIG["DATA_PATH"]
R_PATH = LOCAL_CONFIG["R_PATH"]

RUN_R_SCRIPTS = False
REPLACE_ANALYSIS_DATA = False

ANALYSIS_DATA_FILEPATH = os.path.join(DATA_PATH, "learning_analysis_data.parquet")


In [27]:
# Get analysis data
#
# Important columns:
# - userId, created_at, sats48, num_words, num_img_or_links, is_link_post
#
# Each row is a post

df = dt.get_post_quality_analysis_data()
df = df.loc[df['title'] != 'deleted by author'].reset_index(drop=True)

/Users/ekung/projects/sn-research/src/notebooks/../python/data_tools.py:89: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return x.dt.to_period('W-SAT').dt.start_time
/Users/ekung/projects/sn-research/src/notebooks/../python/data_tools.py:89: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  return x.dt.to_period('W-SAT').dt.start_time


In [28]:
# bin post types into 5 categories:
# 0. Link posts with no text
# 1. Posts with below median words, no images and links in text body
# 2. Posts with above median words, no images and links in text body
# 3. Posts with below median words, images and links in text body
# 4. Posts with above median words, images and links in text body

df['post_type'] = 0

median_words = df.loc[df['num_words']>0, 'num_words'].median()

df.loc[ df['is_link_post'] & (df['num_words']==0), 'post_type'] = 0
df.loc[ (df['num_words']>0) & (df['num_words']<=median_words) & (df['num_img_or_links']==0), 'post_type'] = 1
df.loc[ (df['num_words']>0) & (df['num_words']<=median_words) & (df['num_img_or_links']>0), 'post_type'] = 2
df.loc[ (df['num_words']>0) & (df['num_words']>median_words) & (df['num_img_or_links']==0), 'post_type'] = 3
df.loc[ (df['num_words']>0) & (df['num_words']>median_words) & (df['num_img_or_links']>0), 'post_type'] = 4

categories = sorted(df['post_type'].unique().tolist())   # list of categories
users = df['userId'].unique().tolist()                   # list of users

category_labels = {
    0: "Link-only posts with no post body text", 
    1: r"$\leq$Median words, no images or links in post body", 
    2: r"$\leq$Median words, images or links in post body",
    3: r"$>$Median words, no images or links in post body",
    4: r"$>$Median words, images or links in post body"
}


In [29]:
# latex summary table

header = r"""\begin{table}[H]
\centering
\caption{Post Types Summary}
\vspace{0.2cm}
\label{tbl_post_types_summary}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lrrr}
\toprule
& \makecell{Number\\of posts} & \makecell{Avg. sats\\received in\\first 48 hours} & \makecell{Avg. replies\\received in\\first 48 hours} \\ \midrule
& & & \\
"""
footer = r"""
& & & \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This table reports the number of posts and the average amount of sats and replies received in the first 48 hours, for each post type.}
\end{tablenotes}
\end{threeparttable}
\end{adjustbox}
\end{table}
"""

tbl = ""
for c in categories:
    tbl += f"{category_labels[c]}"
    tbl += f" & {(df['post_type']==c).sum():,.0f}"
    tbl += f" & {df.loc[df['post_type']==c, 'sats48'].mean():,.0f}"
    tbl += f" & {df.loc[df['post_type']==c, 'comments48'].mean():,.1f}"
    tbl += r" \\ [1ex]" + "\n"

print(header + tbl + footer)

with open(os.path.join(LOCAL_PATH, "results", "tbl_post_types_summary.tex"), "w") as f:
    f.write(header + tbl + footer)



\begin{table}[H]
\centering
\caption{Post Types Summary}
\vspace{0.2cm}
\label{tbl_post_types_summary}
\begin{adjustbox}{max width=\textwidth}
\begin{threeparttable}
\begin{tabular}{lrrr}
\toprule
& \makecell{Number\\of posts} & \makecell{Avg. sats\\received in\\first 48 hours} & \makecell{Avg. replies\\received in\\first 48 hours} \\ \midrule
& & & \\
Link-only posts with no post body text & 90,154 & 167 & 1.3 \\ [1ex]
$\leq$Median words, no images or links in post body & 19,385 & 296 & 5.1 \\ [1ex]
$\leq$Median words, images or links in post body & 23,517 & 271 & 3.1 \\ [1ex]
$>$Median words, no images or links in post body & 19,172 & 564 & 6.8 \\ [1ex]
$>$Median words, images or links in post body & 23,629 & 1,112 & 6.9 \\ [1ex]

& & & \\
\bottomrule
\end{tabular}
\begin{tablenotes}
\item {\textit{Notes: } This table reports the number of posts and the average amount of sats and replies received in the first 48 hours, for each post type.}
\end{tablenotes}
\end{threeparttable}
\end{a

In [19]:
# further data prep

df = df.sort_values(by='created_at', ascending=True).reset_index(drop=True)  # sort by timestamp

df['lnsats48'] = np.log1p(df['sats48'])  # log transform sats48

df['ones'] = 1 # create a column of ones


# for each category c, create a column for lnsats48_{c} and is_cat_{c}
# lnsats48_{c} is lnsats48 for posts of category c and zero otherwise
# is_cat_{c} is 1 for posts of category c and zero otherwise
for c in categories:
    df[f'lnsats48_{c}'] = 0.0
    df[f'is_cat_{c}'] = 0
    df.loc[df['post_type']==c, f'lnsats48_{c}'] = df.loc[df['post_type']==c, 'lnsats48']
    df.loc[df['post_type']==c, f'is_cat_{c}'] = 1
    df[f'cum_sum_lnsats48_{c}'] = df[f'lnsats48_{c}'].shift(1).cumsum()  # cumulative sum of lnsats48 for category c posts up to but not including current post
    df[f'cum_count_{c}'] = df[f'is_cat_{c}'].shift(1).cumsum()  # cumulative count of all category c posts up to but not including current post


In [20]:
# mapping of user ids to index of user's first post
user_first_post_idx = {}
for u in users:
    user_first_post_idx[u] = df.loc[df['userId']==u].index[0]

In [21]:
# cum_avg_lnsats48_{c}:
#     For each user i and each category c, the average lnsats48 for all category c posts posted between user i's first appearance date and the current post
#
# i.e. Information set is all posts between the user's first post and the current post

for c in categories:
    df[f'cum_avg_lnsats48_{c}'] = np.nan

for i, u in enumerate(users):
    user_posts = df.loc[df['userId']==u].index
    first_post_idx = user_first_post_idx[u]
    n = len(df)
    selection_mask = (df['created_at'] >= df.loc[first_post_idx, 'created_at']).to_numpy()
    for c in categories:
        lnsats48 = df[f'lnsats48_{c}'].to_numpy() * selection_mask
        is_cat = df[f'is_cat_{c}'].to_numpy() * selection_mask
        numer = np.concat([[np.nan], lnsats48[0:n-1].cumsum()])
        denom = np.concat([[np.nan], is_cat[0:n-1].cumsum()])
        df.loc[user_posts, f'cum_avg_lnsats48_{c}'] = numer[user_posts] / denom[user_posts]


In [22]:
# cum_avg_lnsats48_recent_{c}:
#     For each user i and each category c, the average lnsats48 for all category c posts posted before the current post,
#     weighted by recency decay
#
# i.e. Information set is all posts before the current post, with weights decaying exponentially in time

half_life = pd.Timedelta(days=28)
decay_lambda = np.log(2) / half_life.total_seconds()

for c in categories:
    df[f'cum_avg_lnsats48_recent_{c}'] = np.nan

numer = np.zeros((len(categories), len(df)))  # running weighted sum of lnsats48
denom = np.zeros((len(categories), len(df)))  # running sum of weights
prev_post_time = df.loc[0, 'created_at']
for idx, row in df.iterrows():
    if idx==0:
        continue
    curr_post_time = row['created_at']
    dt = (curr_post_time - prev_post_time).total_seconds()
    decay = np.exp(-decay_lambda * dt)
    for c in categories:
        # since averages don't include current post, check current running numerator and denominator
        if denom[c, idx-1]>0:
            df.loc[idx, f'cum_avg_lnsats48_recent_{c}'] = numer[c, idx-1] / denom[c, idx-1]
        
        # now update running numerator and denominator
        numer[c, idx] = decay * numer[c, idx-1] + row[f'lnsats48_{c}']
        denom[c, idx] = decay * denom[c, idx-1] + row[f'is_cat_{c}']
        
    prev_post_time = curr_post_time


In [23]:
# cum_avg_lnsats48_user_{c}:
#     For each user i and each category c, the average lnsats48 for all category c posts posted by user i, up to the current post
#
# i.e. Information set is the user's own posts

for c in categories:
    df[f'cum_avg_lnsats48_user_{c}'] = np.nan

for i, u in enumerate(users):
    user_posts = df.loc[df['userId']==u].index
    n = len(user_posts)
    for c in categories:
        lnsats48 = df.loc[user_posts, f'lnsats48_{c}'].to_numpy()
        is_cat = df.loc[user_posts, f'is_cat_{c}'].to_numpy()
        numer = np.concat([[np.nan], lnsats48[0:n-1].cumsum()])
        denom = np.concat([[np.nan], is_cat[0:n-1].cumsum()])
        df.loc[user_posts, f'cum_avg_lnsats48_user_{c}'] = numer / denom


In [24]:
# cum_avg_lnsats48_activity_{c}:
#     For each user i and each category c, the average lnsats48 for all category c posts posted between user i's first appearance date and the current post,
#     weighted by proximity to user's peak activity hours
#
# i.e. Information set is all posts between user's first post and the current post, weighted by proximity to user's peak activity hours

PERIOD = 24*3600
HALF_LIFE = 3*3600
SIGMA = HALF_LIFE / np.log(2)

post_times = (df['created_at'].dt.hour * 3600 + df['created_at'].dt.minute * 60 + df['created_at'].dt.second).astype(float).to_numpy()

user_peak_times = {}
for i, u in enumerate(users):
    user_posts = df.loc[df['userId']==u].index
    user_peak_times[u] = post_times[user_posts].mean()

for c in categories:
    df[f'cum_avg_lnsats48_activity_{c}'] = np.nan

for i, u in enumerate(users):
    user_posts = df.loc[df['userId']==u].index
    first_post_idx = user_first_post_idx[u]
    n = len(df)
    selection_mask = (df['created_at'] >= df.loc[first_post_idx, 'created_at']).to_numpy()
    peak_time = user_peak_times[u]
    time_diffs = peak_time - post_times
    time_diffs = (time_diffs + PERIOD/2) % PERIOD - PERIOD/2
    weights = np.exp(-(time_diffs/SIGMA)**2)
    
    for c in categories:
        lnsats48 = df[f'lnsats48_{c}'].to_numpy() * selection_mask * weights
        is_cat = df[f'is_cat_{c}'].to_numpy() * selection_mask * weights
        numer = np.concat([[np.nan], lnsats48[0:n-1].cumsum()])
        denom = np.concat([[np.nan], is_cat[0:n-1].cumsum()])
        df.loc[user_posts, f'cum_avg_lnsats48_activity_{c}'] = numer[user_posts] / denom[user_posts]


In [25]:
# construct long dataframe for conditional logit

df = df.sort_values(by=['userId', 'created_at']).reset_index(drop=True)
df['experience_posts'] = df.groupby('userId').cumcount()
df['experience_days'] = (df['created_at'] - df.groupby('userId')['created_at'].transform('min')) / pd.Timedelta(days=1)

long_df = {
    'userId': [], 'timestamp': [], 'itemId': [], 'weekId': [], 'subId': [], 
    'choice': [], 'post_type': [], 'chosen': [], 'experience_posts': [], 'experience_days': [],
    'cum_avg_lnsats48': [], 'cum_avg_lnsats48_recent': [], 'cum_avg_lnsats48_user': [], 'cum_avg_lnsats48_activity': []
}

for idx, row in df.iterrows():
    u = row['userId']
    t = row['created_at']
    choice = row['post_type']
    for c in categories:
        long_df['userId'].append(u)
        long_df['timestamp'].append(t)
        long_df['itemId'].append(row['itemId'])
        long_df['weekId'].append(row['weekId'])
        long_df['subId'].append(row['subId'])
        long_df['choice'].append(choice)
        long_df['post_type'].append(c)
        long_df['chosen'].append(c==choice)
        long_df['experience_posts'].append(row['experience_posts'])
        long_df['experience_days'].append(row['experience_days'])
        long_df['cum_avg_lnsats48'].append(row[f'cum_avg_lnsats48_{c}'])
        long_df['cum_avg_lnsats48_recent'].append(row[f'cum_avg_lnsats48_recent_{c}'])
        long_df['cum_avg_lnsats48_user'].append(row[f'cum_avg_lnsats48_user_{c}'])
        long_df['cum_avg_lnsats48_activity'].append(row[f'cum_avg_lnsats48_activity_{c}'])

long_df = pd.DataFrame(long_df)

long_df.to_parquet(ANALYSIS_DATA_FILEPATH)
